# Deep Learning Lab 0
In this lab we will be training Convolutional Neural Networks and Visualize the progress through Wandb.
# Task 0.1

## Imports

In [67]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import random_split

import torchvision.models as models
from torchvision.models import alexnet, AlexNet_Weights

import wandb
import os

## Wandb Setup


In [68]:
wandb.init(project="DeepLearning")
#wandb.watch()


## Preprocessing
### Checklist Train Data
- Normalization
- Data Augmentation
- Create images --> ToTensor()
- Dataloader
### Checklist Test Data
- Normalization
- Create images --> ToTensor()

In [69]:
"""
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4), # Randomly Crops 32x32 Region After Padding To Create Small Shifts --> Robustness
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])

"""

transformer = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])

transformer_MNIST = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5), std=(0.5))
    ])

transformer_SVHN = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5), std=(0.5))
])

traindata = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, 
    download=True, 
    transform=transformer)

testdata = torchvision.datasets.CIFAR10(
    root='./data', 
    train=False, 
    download=True, 
    transform=transformer)

/usr/local/lib/python3.11/dist-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [70]:
train_size = int(0.8*len(traindata))
val_size = len(traindata) - train_size

train_dataset, val_dataset = random_split(traindata, [train_size, val_size], generator=torch.Generator().manual_seed(42))

trainload = torch.utils.data.DataLoader(traindata, batch_size=32, shuffle=True)
valload = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
testload = torch.utils.data.DataLoader(testdata, batch_size=32, shuffle=False)

## Training and Testing 
### Checklist
- Activation Function LeakyReLU
- Optimizer Stochastic Gradient Descent lr = 0.0001
- Accuracy on Test set
- Val/Train Stopping when the Val loss converges
- Tensorboard and wandb
- Schedulers


In [6]:
if torch.cuda.is_available():
    device = "cuda" #Nvidia Graphics Card
elif torch.backends.mps.is_available():
    device = "mps" # Apple
else:
    device = "cpu" # Worst Case
print(device)


cuda


In [72]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Images: 32x32
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2), # Images: (32x32)/(2*2) --> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2) # Images  (16x16)/(2x2) --> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), # 4096 --> 128
            nn.LeakyReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

class CNN_MNIST(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN_MNIST, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), # Images: 32x32
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2), # Images: (32x32)/(2*2) --> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2) # Images  (16x16)/(2x2) --> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128), # 4096 --> 128
            nn.LeakyReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
model = CNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)
num_epochs = 10

In [74]:
# ------------------------- Wandb -------------------------


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode="min",
        factor = 0.1, 
        patience = 3 # Wait x amount of epochs before reducing lr
    )

best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        

    average_loss = running_loss / len(trainload)
    wandb.log({
        "Training loss - SGD and LeakyReLu": average_loss
        })
    
    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

            

        average_val_loss = val_running_loss / len(valload)
        wandb.log({
             "validation loss - SGD and LeakyReLU": average_val_loss
             })
    scheduler.step(average_loss)

    # ------------------------- Saving Best Model -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model.state_dict(), "BestModel_SGDReLu.pth")
        print("Saved The Best Performing Model")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

Saved The Best Performing Model
Epoch [1/10] train loss: 2.202, val loss: 2.093, lr: 0.000100
Saved The Best Performing Model
Epoch [2/10] train loss: 2.028, val loss: 1.948, lr: 0.000100
Saved The Best Performing Model
Epoch [3/10] train loss: 1.919, val loss: 1.849, lr: 0.000100
Saved The Best Performing Model
Epoch [4/10] train loss: 1.836, val loss: 1.771, lr: 0.000100
Saved The Best Performing Model
Epoch [5/10] train loss: 1.774, val loss: 1.717, lr: 0.000100
Saved The Best Performing Model
Epoch [6/10] train loss: 1.721, val loss: 1.658, lr: 0.000100
Saved The Best Performing Model
Epoch [7/10] train loss: 1.674, val loss: 1.618, lr: 0.000100
Saved The Best Performing Model
Epoch [8/10] train loss: 1.637, val loss: 1.576, lr: 0.000100
Saved The Best Performing Model
Epoch [9/10] train loss: 1.603, val loss: 1.545, lr: 0.000100
Saved The Best Performing Model
Epoch [10/10] train loss: 1.573, val loss: 1.512, lr: 0.000100


In [75]:
model.load_state_dict(torch.load("BestModel.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()        

average_test_loss = test_loss / len(testload)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

# wandb.finish()


Test loss: 1.509
Test accuracy: 48.76%


### Checklist 2
- LeakyReLU --> Tanh
- Optimizer: SGD --> Adam
- Visualize the results on a Tensorboard

In [76]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Images: 32x32
            nn.BatchNorm2d(32),
            nn.Tanh(),
            nn.MaxPool2d(2,2), # Images: (32x32)/(2*2) --> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.Tanh(),
            nn.MaxPool2d(2,2) # Images  (16x16)/(2x2) --> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), # 4096 --> 128
            nn.Tanh(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [77]:
model = CNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
num_epochs = 10

In [78]:
# ------------------------- Wandb -------------------------


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode="min",
        factor = 0.1, 
        patience = 3 # Wait x amount of epochs before reducing lr
    )

best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        

    average_loss = running_loss / len(trainload)
    wandb.log({
        "Training loss Task AdamTanh": average_loss
        })
    
    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

            

        average_val_loss = val_running_loss / len(valload)
        wandb.log({
             "validation loss Task AdamTanh": average_val_loss
             })
    scheduler.step(average_loss)

    # ------------------------- Saving Best Model -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model.state_dict(), "BestModel_Task_AdamTanh.pth")
        print("Saved The Best Performing Model")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

Saved The Best Performing Model
Epoch [1/10] train loss: 1.453, val loss: 1.161, lr: 0.000100
Saved The Best Performing Model
Epoch [2/10] train loss: 1.124, val loss: 0.997, lr: 0.000100
Saved The Best Performing Model
Epoch [3/10] train loss: 0.990, val loss: 0.887, lr: 0.000100
Saved The Best Performing Model
Epoch [4/10] train loss: 0.904, val loss: 0.788, lr: 0.000100
Saved The Best Performing Model
Epoch [5/10] train loss: 0.836, val loss: 0.728, lr: 0.000100
Saved The Best Performing Model
Epoch [6/10] train loss: 0.776, val loss: 0.702, lr: 0.000100
Saved The Best Performing Model
Epoch [7/10] train loss: 0.727, val loss: 0.650, lr: 0.000100
Saved The Best Performing Model
Epoch [8/10] train loss: 0.683, val loss: 0.615, lr: 0.000100
Saved The Best Performing Model
Epoch [9/10] train loss: 0.641, val loss: 0.563, lr: 0.000100
Saved The Best Performing Model
Epoch [10/10] train loss: 0.600, val loss: 0.506, lr: 0.000100


In [79]:
model.load_state_dict(torch.load("BestModel_Task_AdamTanh.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()         

average_test_loss = test_loss / len(testload)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")



Test loss: 0.863
Test accuracy: 69.65%


# Task 0.2

## 0.2 Task
### 0.2.1 Transfer Learning from ImageNet
- Download and prepare CIFAR-10 dataset (it is already available in the above mentioned libraries)
- Use AlexNet as the model (Pytorch AlexNet)
- You have to perform two separate experiments-
    - Train the model for CIFAR-10 data, Report the test test accuracy. (also referred as fine tuning the model)
    - Use the pretarined weights of AlexNet, in other words use AlexNet as a pretrained network for image classification on CIFAR-10 data (also referred as Feature Extraction), Report the test test accuracy. (optional)
- In both the above cases remember to add an extra fully connected layer to the classifier with number of neurons = 10, because there are 10 classes in CIFAR-10 dataset. This layer will be trainable in both cases.
- Explain (briefly!) what is the difference between the two runs and why there is a difference in performance. (optional)

In [ ]:
epochs = 10
learning_rate = 0.0001

transformer = transforms.Compose([
    transforms.Resize((224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])  

# Download CIFAR-10 datasets
trainset_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transformer)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transformer)

trainfull_size = len(trainset_full)

train_size = int(0.7 * trainfull_size)
val_size = int(0.3 * trainfull_size)

trainset, valset = random_split(
    trainset_full, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
valloader = torch.utils.data.DataLoader(valset, batch_size=64, shuffle=False)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

print(f"Dataset Split - Train: {train_size}, Val: {val_size}, Test: {len(testset)}")


print("\n" + "=" * 50)
print("EXPERIMENT 1: Fine-Tuning Pretrained AlexNet")
print("=" * 50)

model_finetune = models.alexnet(pretrained=True)
# Replace the last fully connected layer to output 10 classes (CIFAR-10)
model_finetune.classifier[6] = nn.Linear(4096, 10)
model_finetune = model_finetune.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_finetune = torch.optim.Adam(model_finetune.parameters(), lr=learning_rate)

# Training loop for fine-tuning
best_val_loss = float('inf')
for epoch in range(epochs):
    model_finetune.train()
    running_loss = 0.0
    
    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer_finetune.zero_grad()
        
        outputs = model_finetune(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer_finetune.step()
        
        running_loss += loss.item()
    
    avg_train_loss = running_loss / len(trainloader)
    wandb.log({
        "Training loss - Fine Tuning": avg_train_loss
        })


    # Validation
    model_finetune.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in valloader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model_finetune(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(valloader)
    wandb.log({
        "Validation Loss - Fine Tuning": avg_val_loss
        })
    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model_finetune.state_dict(), "BestModel_Finetune.pth")

# Test the fine-tuned model
model_finetune.load_state_dict(torch.load("BestModel_Finetune.pth"))
model_finetune.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model_finetune(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_finetune = 100 * (correct / total)
print(f"\nTest Accuracy (Fine-tuning): {accuracy_finetune:.2f}%")
# wandb.log({"finetune_test_accuracy": accuracy_finetune})

Files already downloaded and verified


/home/juulg/anaconda3/envs/torch_gpu/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Files already downloaded and verified
Dataset Split - Train: 35000, Val: 15000, Test: 10000

EXPERIMENT 1: Fine-Tuning Pretrained AlexNet


/home/juulg/anaconda3/envs/torch_gpu/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/juulg/anaconda3/envs/torch_gpu/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


KeyboardInterrupt: 

In [81]:
# ======================== EXPERIMENT 2: Feature Extraction (Frozen Pretrained AlexNet) ========================
print("\n" + "=" * 50)
print("EXPERIMENT 2: Feature Extraction (Frozen AlexNet)")
print("=" * 50)

model_feature_extract = models.alexnet(pretrained=True)

# Freeze all layers except the final classifier layer
for param in model_feature_extract.features.parameters():
    param.requires_grad = False

# Only the newly added layer will be trainable
model_feature_extract.classifier[6] = nn.Linear(4096, 10)
model_feature_extract = model_feature_extract.to(device)

# Only optimize the new classifier layer
optimizer_feature_extract = torch.optim.Adam(model_feature_extract.classifier[6].parameters(), lr=learning_rate)

# Training loop for feature extraction
best_val_loss_fe = float('inf')
for epoch in range(epochs):
    model_feature_extract.train()
    running_loss = 0.0
    
    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer_feature_extract.zero_grad()
        
        outputs = model_feature_extract(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer_feature_extract.step()
        
        running_loss += loss.item()
    
    avg_train_loss = running_loss / len(trainloader)
    wandb.log({
        "Training loss - Feature Extraction": avg_train_loss
        })

    # Validation
    model_feature_extract.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in valloader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model_feature_extract(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(valloader)
    
    wandb.log({
        "Validation loss - Feature Extraction": avg_val_loss
        })
    
    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # Save best model
    if avg_val_loss < best_val_loss_fe:
        best_val_loss_fe = avg_val_loss
        torch.save(model_feature_extract.state_dict(), "BestModel_FeatureExtract.pth")

# Test the feature extraction model
model_feature_extract.load_state_dict(torch.load("BestModel_FeatureExtract.pth"))
model_feature_extract.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model_feature_extract(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_feature_extract = 100 * (correct / total)
print(f"\nTest Accuracy (Feature Extraction): {accuracy_feature_extract:.2f}%")
wandb.log({"feature_extract_test_accuracy": accuracy_feature_extract})

# ======================== COMPARISON ========================
print("\n" + "=" * 50)
print("COMPARISON SUMMARY")
print("=" * 50)
print(f"Fine-Tuning Test Accuracy:      {accuracy_finetune:.2f}%")
print(f"Feature Extraction Accuracy:    {accuracy_feature_extract:.2f}%")
print(f"Difference:                     {abs(accuracy_finetune - accuracy_feature_extract):.2f}%")
print("\nExplanation:")
print("- Fine-tuning allows all layers to be updated with CIFAR-10 data,")
print("  adapting the learned ImageNet features to the new task.")
print("- Feature extraction keeps the pre-trained weights frozen,")
print("  using ImageNet features as fixed representations.")
print("- Fine-tuning typically performs better as it can specialize")
print("  the model filters to the specific CIFAR-10 task.")
# wandb.finish()


EXPERIMENT 2: Feature Extraction (Frozen AlexNet)
Epoch [1/10] Train Loss: 1.1643 | Val Loss: 0.8650
Epoch [2/10] Train Loss: 0.8779 | Val Loss: 0.7668
Epoch [3/10] Train Loss: 0.8142 | Val Loss: 0.7185
Epoch [4/10] Train Loss: 0.7851 | Val Loss: 0.6970
Epoch [5/10] Train Loss: 0.7626 | Val Loss: 0.6729
Epoch [6/10] Train Loss: 0.7491 | Val Loss: 0.6599
Epoch [7/10] Train Loss: 0.7360 | Val Loss: 0.6502
Epoch [8/10] Train Loss: 0.7231 | Val Loss: 0.6446
Epoch [9/10] Train Loss: 0.7172 | Val Loss: 0.6323
Epoch [10/10] Train Loss: 0.7098 | Val Loss: 0.6262

Test Accuracy (Feature Extraction): 77.69%

COMPARISON SUMMARY
Fine-Tuning Test Accuracy:      90.96%
Feature Extraction Accuracy:    77.69%
Difference:                     13.27%

Explanation:
- Fine-tuning allows all layers to be updated with CIFAR-10 data,
  adapting the learned ImageNet features to the new task.
- Feature extraction keeps the pre-trained weights frozen,
  using ImageNet features as fixed representations.
- Fine-t

### Task 0.2.2

Prepare a CNN of your choice and train it on the MNIST data. Report the accuracy

In [82]:
#Task 2.2
traindata_MNIST = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transformer_MNIST)

testdata_MNIST = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transformer_MNIST)


train_size = int(0.8*len(traindata_MNIST))
val_size = len(traindata_MNIST) - train_size

train_dataset_MNIST, val_dataset_MNIST = random_split(traindata_MNIST, [train_size, val_size], generator=torch.Generator().manual_seed(42))

trainload_MNIST = torch.utils.data.DataLoader(traindata_MNIST, batch_size=32, shuffle=True)
valload_MNIST = torch.utils.data.DataLoader(val_dataset_MNIST, batch_size=32, shuffle=False)
testload_MNIST = torch.utils.data.DataLoader(testdata_MNIST, batch_size=32, shuffle=False)

model = CNN_MNIST(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)
num_epochs = 10

In [83]:
best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload_MNIST:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(trainload_MNIST)
    wandb.log({
        "Training loss - MNIST": average_loss
        })
    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload_MNIST:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        average_val_loss = val_running_loss / len(valload_MNIST)

        wandb.log({
            "Validation loss - MNIST": average_val_loss
        })

    # scheduler.step(average_loss)

    # ------------------------- Saving Best Model -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model.state_dict(), "BestModel_MNIST.pth")
        print("Saved The Best Performing Model")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

Saved The Best Performing Model
Epoch [1/10] train loss: 1.828, val loss: 1.364, lr: 0.000100
Saved The Best Performing Model
Epoch [2/10] train loss: 1.078, val loss: 0.836, lr: 0.000100
Saved The Best Performing Model
Epoch [3/10] train loss: 0.721, val loss: 0.597, lr: 0.000100
Saved The Best Performing Model
Epoch [4/10] train loss: 0.551, val loss: 0.470, lr: 0.000100
Saved The Best Performing Model
Epoch [5/10] train loss: 0.452, val loss: 0.400, lr: 0.000100
Saved The Best Performing Model
Epoch [6/10] train loss: 0.389, val loss: 0.345, lr: 0.000100
Saved The Best Performing Model
Epoch [7/10] train loss: 0.346, val loss: 0.310, lr: 0.000100
Saved The Best Performing Model
Epoch [8/10] train loss: 0.313, val loss: 0.281, lr: 0.000100
Saved The Best Performing Model
Epoch [9/10] train loss: 0.286, val loss: 0.258, lr: 0.000100
Saved The Best Performing Model
Epoch [10/10] train loss: 0.265, val loss: 0.239, lr: 0.000100


In [84]:
#Test best model for MNIST
model.load_state_dict(torch.load("BestModel_MNIST.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload_MNIST:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()        

average_test_loss = test_loss / len(testload_MNIST)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")


Test loss: 0.220
Test accuracy: 94.79%


Use the above model as a pre-trained CNN for the SVHN dataset. Report the accuracy

In [85]:
#Import SVHN Dataset 0.2.2
traindata_SVHN = torchvision.datasets.SVHN(
    root='./data',
    split="train",
    download=True,
    transform=transformer_SVHN)

testdata_SVHN = torchvision.datasets.SVHN(
    root='./data',
    split="test",
    download=True,
    transform=transformer_SVHN)


train_size = int(0.8*len(traindata_SVHN))
val_size = len(traindata_SVHN) - train_size

train_dataset_SVHN, val_dataset_SVHN = random_split(traindata_SVHN, [train_size, val_size], generator=torch.Generator().manual_seed(42))

trainload_SVHN = torch.utils.data.DataLoader(traindata_SVHN, batch_size=32, shuffle=True)
valload_SVHN = torch.utils.data.DataLoader(val_dataset_SVHN, batch_size=32, shuffle=False)
testload_SVHN = torch.utils.data.DataLoader(testdata_SVHN, batch_size=32, shuffle=False)

In [86]:
#Test MNIST model on SVHN dataset

#Test best model for MNIST
model.load_state_dict(torch.load("BestModel_MNIST.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload_SVHN:
        images = images.to(device)
        if images.shape[1] == 3:  # RGB
            images = images.mean(dim=1, keepdim=True)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

average_test_loss = test_loss / len(testload_SVHN)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

# wandb.finish()


Test loss: 2.222
Test accuracy: 20.46%


In the third step you are performing transfer learning from MNIST to SVHN.

In [87]:
#Transfer learning from MNIST to SVHN 
model.load_state_dict(torch.load("BestModel_MNIST.pth"))
model.to(device)

#Freeze model weights
for param in model.parameters():
    param.requires_grad = False

model.classifier[4] = nn.Linear(128, 10)
model = model.to(device) 
optimizer = torch.optim.SGD(model.classifier[4].parameters(), lr=0.0001)

best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload_SVHN:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(trainload_SVHN)
    wandb.log({
        "Training loss - Transfer Learning MNIST MODEL on SVHN Dataset": average_loss
        })

    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload_SVHN:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        average_val_loss = val_running_loss / len(valload_SVHN)

        wandb.log({
            "Validation loss - Transfer Learning MNIST MODEL on SVHN Dataset": average_val_loss
        })

    scheduler.step(average_loss)

    # ------------------------- Saving Best Model -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model.state_dict(), "BestModel_SVHN_Transfer.pth")
        print("Saved The Best Performing Model")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

Saved The Best Performing Model
Epoch [1/10] train loss: 2.341, val loss: 2.259, lr: 0.000100
Saved The Best Performing Model
Epoch [2/10] train loss: 2.265, val loss: 2.227, lr: 0.000100
Saved The Best Performing Model
Epoch [3/10] train loss: 2.236, val loss: 2.203, lr: 0.000100
Saved The Best Performing Model
Epoch [4/10] train loss: 2.215, val loss: 2.183, lr: 0.000100
Saved The Best Performing Model
Epoch [5/10] train loss: 2.194, val loss: 2.164, lr: 0.000100
Saved The Best Performing Model
Epoch [6/10] train loss: 2.175, val loss: 2.146, lr: 0.000100
Saved The Best Performing Model
Epoch [7/10] train loss: 2.159, val loss: 2.130, lr: 0.000100
Saved The Best Performing Model
Epoch [8/10] train loss: 2.143, val loss: 2.115, lr: 0.000100
Saved The Best Performing Model
Epoch [9/10] train loss: 2.132, val loss: 2.100, lr: 0.000100
Saved The Best Performing Model
Epoch [10/10] train loss: 2.118, val loss: 2.092, lr: 0.000100


In [88]:
#Test transfer model on SVHN dataset

#Test best model for MNIST
model.load_state_dict(torch.load("BestModel_SVHN_Transfer.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload_SVHN:
        images = images.to(device)
        if images.shape[1] == 3:  # RGB
            images = images.mean(dim=1, keepdim=True)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

average_test_loss = test_loss / len(testload_SVHN)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

wandb.finish()

Test loss: 2.085
Test accuracy: 28.76%


Training loss - Feature Extraction,█▄▃▂▂▂▁▁▁▁
Training loss - Fine Tuning,█▄▃▂▂▂▁▁▁▁
Training loss - MNIST,█▅▃▂▂▂▁▁▁▁
Training loss - SGD and LeakyReLu,█▆▅▄▃▃▂▂▁▁
Training loss - Transfer Learning MNIST MODEL on SVHN Dataset,█▆▅▄▃▃▂▂▁▁
Training loss Task AdamTanh,█▅▄▃▃▂▂▂▁▁
Validation Loss - Fine Tuning,█▂▃▁▁▄▅▅█▅
Validation loss - Feature Extraction,█▅▄▃▂▂▂▂▁▁
Validation loss - MNIST,█▅▃▂▂▂▁▁▁▁
Validation loss - Transfer Learning MNIST MODEL on SVHN Dataset,█▇▆▅▄▃▃▂▁▁
+3,...
